# Revolving-Credit Default: Feature Engineering

This notebook turns the source hypotheses from notebook 01 into reusable Feature Store definitions, then generates immutable point-in-time Datasets. It does not fit models.

Run section by section. Review each intermediate result before registering its Feature View.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import yaml
from IPython.display import display
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as sf
from snowflake.snowpark.window import Window
from snowflake.ml.feature_store import FeatureStore, CreationMode, Entity, FeatureView, Feature

session = get_active_session()
config_path = Path("../project.yaml")
if not config_path.is_file():
    config_path = Path("project.yaml")
config = yaml.safe_load(config_path.read_text())

database_name = config["snowflake"]["database"]
raw_schema = config["snowflake"]["schemas"]["raw"]
dev_schema = config["snowflake"]["schemas"]["dev"]
feature_schema = config["snowflake"]["schemas"]["feature_store"]
session.use_role(config["snowflake"]["roles"]["developer"])
session.use_warehouse(config["snowflake"]["warehouse"])
session.use_database(database_name)
session.use_schema(dev_schema)

fs = FeatureStore(session=session, database=database_name, name=feature_schema,
                  default_warehouse=config["snowflake"]["warehouse"],
                  creation_mode=CreationMode.CREATE_IF_NOT_EXIST)

## 1. Establish the entity after confirming the grain

Notebook 01 established one account per facility and monthly prediction observations. `ACCOUNT_ID` is therefore the immutable Feature Store join key.

In [ ]:
entity_name = "CREDIT_ACCOUNT"
entity_exists = fs.list_entities().filter(sf.col("NAME") == entity_name).count() > 0
if not entity_exists:
    fs.register_entity(Entity(
        name=entity_name,
        join_keys=["ACCOUNT_ID"],
        desc="Existing revolving-credit facility used as the Feature Store entity.",
    ))
credit_account = fs.get_entity(name=entity_name)
actual_join_keys = [key.resolved() for key in credit_account.join_keys]
if actual_join_keys != ["ACCOUNT_ID"]:
    raise ValueError(f"Unexpected entity keys: {actual_join_keys}")
fs.list_entities().filter(sf.col("NAME") == entity_name).show()

## 2. Develop account profile features

Profile features change slowly. Income uses the dated customer snapshot, and limit-change recency uses only events effective by the feature timestamp. Ratios use `DIV0NULL` so an invalid denominator remains visible as missing rather than becoming zero.

In [ ]:
accounts = session.table(f"{database_name}.{raw_schema}.DIM_CREDIT_ACCOUNT")
customers = session.table(f"{database_name}.{raw_schema}.DIM_CUSTOMER")
financials = session.table(f"{database_name}.{raw_schema}.FACT_CUSTOMER_FINANCIAL_SNAPSHOT")
events = session.table(f"{database_name}.{raw_schema}.FACT_ACCOUNT_EVENT")
daily = session.table(f"{database_name}.{raw_schema}.FACT_ACCOUNT_DAILY_SNAPSHOT")

limit_events = (events.filter(sf.col("EVENT_TYPE") == "CREDIT_LIMIT_CHANGE")
                    .select("ACCOUNT_ID", sf.to_date("EVENT_TIMESTAMP").alias("EVENT_DATE")
                        , sf.col("OLD_VALUE").cast("double").alias("OLD_LIMIT")
                        , sf.col("NEW_VALUE").cast("double").alias("NEW_LIMIT"),)
                )

monthly_daily = daily.filter(sf.dayofmonth(sf.dateadd("day", sf.lit(1), "SNAPSHOT_DATE")) == 1)

profile_candidates = (monthly_daily.join(accounts, "ACCOUNT_ID")
                                .join(customers, "CUSTOMER_ID")
                                .join(financials
                                    , (customers["CUSTOMER_ID"] == financials["CUSTOMER_ID"]) 
                                        & (financials["SNAPSHOT_DATE"] <= monthly_daily["SNAPSHOT_DATE"])
                                    ,)
                                .join(limit_events
                                    , (monthly_daily["ACCOUNT_ID"] == limit_events["ACCOUNT_ID"]) 
                                        & (limit_events["EVENT_DATE"] <= monthly_daily["SNAPSHOT_DATE"])
                                    , how="left"
                                    ,)
                                .select(monthly_daily["ACCOUNT_ID"].alias("ACCOUNT_ID")
                                        , monthly_daily["SNAPSHOT_DATE"].alias("MNTH_DAY_SNAPSHOT_DATE")
                                        , "EVENT_DATE", "PRODUCT_CODE", "ORIGINATION_CHANNEL"
                                        , "OPEN_DATE", "CUSTOMER_SINCE_DATE"
                                        , monthly_daily["CREDIT_LIMIT"].alias("CREDIT_LIMIT")
                                        , "MONTHLY_INCOME_ESTIMATE", "NEW_LIMIT", "OLD_LIMIT")
                    )

latest_available = (Window
                    .partition_by("ACCOUNT_ID", "MNTH_DAY_SNAPSHOT_DATE")
                    .order_by(sf.col("MNTH_DAY_SNAPSHOT_DATE").desc(), sf.col("EVENT_DATE").desc_nulls_last())
                    )

profile_df = (profile_candidates
                .with_column("AVAILABILITY_ORDER", sf.row_number().over(latest_available))
                .filter(sf.col("AVAILABILITY_ORDER") == 1)
                .select("ACCOUNT_ID", sf.dateadd("day", sf.lit(1), sf.col("MNTH_DAY_SNAPSHOT_DATE")).cast("timestamp_ntz").alias("PROFILE_TS")
                        , "PRODUCT_CODE", "ORIGINATION_CHANNEL"
                        , sf.datediff("month", sf.col("OPEN_DATE"), sf.col("MNTH_DAY_SNAPSHOT_DATE")).alias("ACCOUNT_AGE_MONTHS")
                        , sf.datediff("month", sf.col("CUSTOMER_SINCE_DATE")
                            , sf.col("MNTH_DAY_SNAPSHOT_DATE")).alias("CUSTOMER_TENURE_MONTHS")
                        , sf.col("CREDIT_LIMIT").alias("CURRENT_CREDIT_LIMIT")
                        , sf.col("MONTHLY_INCOME_ESTIMATE").alias("CURRENT_INCOME_ESTIMATE")
                        , sf.call_function("DIV0NULL", sf.col("CREDIT_LIMIT"), sf.col("MONTHLY_INCOME_ESTIMATE")).alias("LIMIT_TO_INCOME_RATIO")
                        , sf.datediff("month", sf.col("EVENT_DATE")
                            , sf.col("MNTH_DAY_SNAPSHOT_DATE")).alias("MONTHS_SINCE_LIMIT_CHANGE")
                        , sf.iff(sf.col("EVENT_DATE") >= sf.dateadd("month", sf.lit(-12),sf.col("MNTH_DAY_SNAPSHOT_DATE"))
                                    , sf.call_function("DIV0NULL", sf.col("NEW_LIMIT") - sf.col("OLD_LIMIT"), sf.col("OLD_LIMIT"))
                                    , sf.lit(0.0), ).alias("LIMIT_CHANGE_PERCENT_12M")
                                ,)
                )

profile_df.sort("PROFILE_TS", "ACCOUNT_ID").show(10)

The first-day monthly rows provide the point-in-time profile. Null limit-change features mean no prior change, which is a distinct and meaningful state for model preprocessing.

## 3. Develop balance and delinquency features

Daily history supports true calendar windows. The rolling frame below is day-based because the source has one row per account and calendar date; the current row is the observation-time servicing state.

In [ ]:
order_day = sf.datediff("day", sf.lit("1970-01-01"), sf.col("SNAPSHOT_DATE"))
window_90d = Window.partition_by("ACCOUNT_ID").order_by(order_day).range_between(-89, 0)
window_180d = Window.partition_by("ACCOUNT_ID").order_by(order_day).range_between(-179, 0)
account_order = Window.partition_by("ACCOUNT_ID").order_by("SNAPSHOT_DATE")

servicing = (daily
                .with_column("UTILISATION", sf.call_function("DIV0NULL", sf.col("OUTSTANDING_BALANCE"), sf.col("CREDIT_LIMIT")))
                .with_column("DELINQUENCY_START"
                                , sf.iff((sf.col("DAYS_PAST_DUE") > 0) 
                                        & (sf.coalesce(sf.lag("DAYS_PAST_DUE").over(account_order), sf.lit(0)) == 0)
                                , 1, 0, ))
                .with_column("LATEST_DELINQUENT_DATE"
                            , sf.max(sf.iff(sf.col("DAYS_PAST_DUE") > 0, sf.col("SNAPSHOT_DATE"), sf.lit(None)))
                                    .over(account_order.rows_between(Window.UNBOUNDED_PRECEDING, Window.CURRENT_ROW)))
            )
            
balance_df = (servicing
                .select("ACCOUNT_ID"
                        , sf.dateadd("day", sf.lit(1), sf.col("SNAPSHOT_DATE")).cast("timestamp_ntz").alias("BALANCE_TS")
                        , sf.col("OUTSTANDING_BALANCE").alias("CURRENT_BALANCE")
                        , sf.col("UTILISATION").alias("CURRENT_UTILISATION")
                        , sf.stddev("UTILISATION").over(window_90d).alias("UTILISATION_STDDEV_90D")
                        , (sf.col("OUTSTANDING_BALANCE") - sf.lag("OUTSTANDING_BALANCE", 30).over(account_order)).alias("BALANCE_CHANGE_30D")
                        , (sf.col("OUTSTANDING_BALANCE") - sf.lag("OUTSTANDING_BALANCE", 90).over(account_order)).alias("BALANCE_CHANGE_90D")
                        , sf.call_function("DIV0NULL", sf.col("AVAILABLE_CREDIT"), sf.col("CREDIT_LIMIT")).alias("AVAILABLE_CREDIT_RATIO")
                    )
                .filter(sf.dayofmonth("BALANCE_TS") == 1)
            )

delinquency_df = (servicing
                    .select("ACCOUNT_ID"
                            , sf.dateadd("day", sf.lit(1), sf.col("SNAPSHOT_DATE")).cast("timestamp_ntz").alias("DELINQUENCY_TS")
                            , sf.col("DAYS_PAST_DUE").alias("CURRENT_DAYS_PAST_DUE")
                            , sf.sum("DELINQUENCY_START").over(window_180d).alias("DELINQUENCY_EPISODES_6M")
                            , sf.ceil(sf.col("DAYS_PAST_DUE") / sf.lit(30)).alias("CONSECUTIVE_DELINQUENT_MONTHS")
                            , sf.datediff("day", sf.col("LATEST_DELINQUENT_DATE"), sf.col("SNAPSHOT_DATE")).alias("DAYS_SINCE_LAST_DELINQUENCY")
                        )
                    .filter(sf.dayofmonth("DELINQUENCY_TS") == 1)
                )

balance_df.sort("BALANCE_TS", "ACCOUNT_ID").show(10)
delinquency_df.sort("DELINQUENCY_TS", "ACCOUNT_ID").show(10)

### Use native tiled aggregation for reusable rolling windows

The daily source supports declarative Feature Store aggregation via the `Feature` class. A one-day tile divides all requested windows (30, 90, 180, 365 days) exactly. Rolling averages, maxima, and indicator-based sums that were previously manual Snowpark window functions now live here as tiled aggregations, leaving only lag-based, stddev, and row-level features in the Snowpark cell above.

In [ ]:
aggregation_source = daily.select("ACCOUNT_ID"
                                    , sf.dateadd("day", sf.lit(1), sf.col("SNAPSHOT_DATE")).cast("timestamp_ntz").alias("AGGREGATION_TS")
                                    , sf.call_function("DIV0NULL", sf.col("OUTSTANDING_BALANCE"), sf.col("CREDIT_LIMIT")).alias("DAILY_UTILISATION")
                                    , sf.col("DAYS_PAST_DUE").cast("double").alias("DAILY_DAYS_PAST_DUE")
                                    , sf.iff(sf.col("OUTSTANDING_BALANCE") > sf.col("CREDIT_LIMIT")
                                            , sf.lit(1.0), sf.lit(0.0)).alias("IS_OVER_LIMIT")
                                    , sf.iff(sf.col("DAYS_PAST_DUE") > 0
                                            , sf.lit(1.0), sf.lit(0.0)).alias("IS_DELINQUENT"))

aggregation_features = [
    Feature.avg("DAILY_UTILISATION", "30d").alias("AVG_UTILISATION_30D"),
    Feature.avg("DAILY_UTILISATION", "90d").alias("AVG_UTILISATION_90D"),
    Feature.max("DAILY_UTILISATION", "90d").alias("MAX_UTILISATION_90D"),
    Feature.avg("DAILY_UTILISATION", "365d").alias("AVG_UTILISATION_365D"),
    Feature.max("DAILY_UTILISATION", "365d").alias("MAX_UTILISATION_365D"),
    Feature.max("DAILY_DAYS_PAST_DUE", "90d").alias("MAX_DAYS_PAST_DUE_3M"),
    Feature.max("DAILY_DAYS_PAST_DUE", "180d").alias("MAX_DAYS_PAST_DUE_6M"),
    Feature.max("DAILY_DAYS_PAST_DUE", "365d").alias("MAX_DAYS_PAST_DUE_365D"),
    Feature.sum("IS_OVER_LIMIT", "30d").alias("DAYS_OVER_LIMIT_30D"),
    Feature.sum("IS_DELINQUENT", "90d").alias("DELINQUENT_DAYS_90D"),
]
aggregation_view = FeatureView(
    name="ACCOUNT_AGGREGATION_FV",
    entities=[credit_account],
    feature_df=aggregation_source,
    feature_granularity="1 day",
    features=aggregation_features,
    timestamp_col="AGGREGATION_TS",
    refresh_freq="1 day",
    desc="Tiled daily account behaviour aggregated over 30-, 90-, 180-, and 365-day windows.",
)
print("Native aggregation features:", aggregation_features)
aggregation_source.sort("AGGREGATION_TS", "ACCOUNT_ID").show(10)

## 4. Develop payment and contact features

Event windows require an observation spine. Joining events only when their event date is within the trailing interval makes the inclusion boundary explicit and auditable.

In [ ]:
payments = session.table(f"{database_name}.{raw_schema}.FACT_PAYMENT")
contacts = session.table(f"{database_name}.{raw_schema}.FACT_CUSTOMER_CONTACT")

observation_dates = (session.table(f"{database_name}.{raw_schema}.ACCOUNT_OBSERVATION")
                        .select("ACCOUNT_ID", sf.col("OBSERVATION_DATE").alias("FEATURE_DATE"))
                        .distinct()
                    )
prior_servicing = daily.select("ACCOUNT_ID"
                                , sf.dateadd("day", sf.lit(1), sf.col("SNAPSHOT_DATE")).alias("FEATURE_DATE")
                                , sf.col("OUTSTANDING_BALANCE").alias("PRIOR_BALANCE")
                                , sf.col("AMOUNT_DUE").alias("PRIOR_AMOUNT_DUE"))

payment_events = observation_dates.join(payments
                                        , (observation_dates["ACCOUNT_ID"] == payments["ACCOUNT_ID"])
                                            & (payments["PAYMENT_DATE"] < observation_dates["FEATURE_DATE"])
                                            & (payments["PAYMENT_DATE"] > sf.dateadd("day", sf.lit(-180), observation_dates["FEATURE_DATE"]))
                                        , how="left")

payment_aggregates = (payment_events
                        .group_by(observation_dates["ACCOUNT_ID"]
                                    , observation_dates["FEATURE_DATE"])
                        .agg(sf.sum(sf.iff(sf.datediff("day", sf.col("PAYMENT_DATE"), sf.col("FEATURE_DATE")) < 30
                                            , sf.col("PAYMENT_AMOUNT"), 0)).alias("PAYMENT_SUM_30D")
                            , sf.sum(sf.iff(sf.datediff("day", sf.col("PAYMENT_DATE"), sf.col("FEATURE_DATE")) < 90
                                            , sf.col("PAYMENT_AMOUNT"), 0)).alias("PAYMENT_SUM_90D")
                            , sf.count_if(sf.datediff("day", sf.col("PAYMENT_DATE"), sf.col("FEATURE_DATE")) < 90).alias("PAYMENT_COUNT_90D")
                            , sf.count_if((sf.col("PAYMENT_STATUS") == "MISSED")
                                            & (sf.datediff("day", sf.col("PAYMENT_DATE"), sf.col("FEATURE_DATE")) < 90)).alias("MISSED_PAYMENT_COUNT_3M")
                            , sf.count_if(sf.col("PAYMENT_STATUS") == "MISSED").alias("MISSED_PAYMENT_COUNT_6M")
                            , sf.datediff("day"
                                        , sf.max(sf.iff(sf.col("PAYMENT_AMOUNT") > 0, sf.col("PAYMENT_DATE"), sf.lit(None)))
                                        , sf.col("FEATURE_DATE")).alias("DAYS_SINCE_LAST_PAYMENT")
                            , sf.avg(sf.iff(sf.datediff("day", sf.col("PAYMENT_DATE"), sf.col("FEATURE_DATE")) < 30
                                            , sf.col("PAYMENT_AMOUNT"), sf.lit(None))).alias("AVG_PAYMENT_RECENT_30D")
                            , sf.avg(sf.iff((sf.datediff("day", sf.col("PAYMENT_DATE"), sf.col("FEATURE_DATE")) >= 60)
                                                & (sf.datediff("day", sf.col("PAYMENT_DATE"), sf.col("FEATURE_DATE")) < 90)
                                            , sf.col("PAYMENT_AMOUNT"), sf.lit(None))).alias("AVG_PAYMENT_EARLY_30D"))
                    )
payment_df = (payment_aggregates
                .join(prior_servicing, ["ACCOUNT_ID", "FEATURE_DATE"], how="left")
                .select("ACCOUNT_ID"
                        , sf.col("FEATURE_DATE").cast("timestamp_ntz").alias("PAYMENT_TS")
                        , "PAYMENT_SUM_30D"
                        , "PAYMENT_SUM_90D"
                        , "PAYMENT_COUNT_90D"
                        , sf.call_function("DIV0NULL", "PAYMENT_SUM_30D", "PRIOR_BALANCE").alias("PAYMENT_TO_BALANCE_RATIO_30D")
                        , sf.call_function("DIV0NULL", "PAYMENT_SUM_30D", "PRIOR_AMOUNT_DUE").alias("PAYMENT_TO_AMOUNT_DUE_RATIO_30D")
                        , "MISSED_PAYMENT_COUNT_3M"
                        , "MISSED_PAYMENT_COUNT_6M"
                        , "DAYS_SINCE_LAST_PAYMENT"
                        , (sf.col("AVG_PAYMENT_RECENT_30D") - sf.col("AVG_PAYMENT_EARLY_30D")).alias("PAYMENT_AMOUNT_TREND_3M"))
            )

contact_df = (observation_dates
                .join(contacts
                        , (observation_dates["ACCOUNT_ID"] == contacts["ACCOUNT_ID"])
                            & (sf.to_date(contacts["CONTACT_TIMESTAMP"]) < observation_dates["FEATURE_DATE"])
                            & (sf.to_date(contacts["CONTACT_TIMESTAMP"]) > sf.dateadd("day", sf.lit(-90), observation_dates["FEATURE_DATE"]))
                        , how="left")
                .group_by(observation_dates["ACCOUNT_ID"], "FEATURE_DATE")
                .agg(sf.count_if(sf.datediff("day", sf.to_date("CONTACT_TIMESTAMP"), sf.col("FEATURE_DATE")) < 30).alias("CONTACT_COUNT_30D")
                    , sf.count("CONTACT_ID").alias("CONTACT_COUNT_90D")
                    , sf.count_if(sf.col("CONTACT_OUTCOME") == "BROKEN_PROMISE").alias("BROKEN_PROMISE_COUNT_90D"))
                .select("ACCOUNT_ID"
                        , sf.col("FEATURE_DATE").cast("timestamp_ntz").alias("CONTACT_TS")
                        , "CONTACT_COUNT_30D"
                        , "CONTACT_COUNT_90D"
                        , "BROKEN_PROMISE_COUNT_90D")
            )
payment_df.sort("PAYMENT_TS", "ACCOUNT_ID").show(10)
contact_df.sort("CONTACT_TS", "ACCOUNT_ID").show(10)

### Validate selected accounts before registration

Use a small mix of facilities with and without default events. Compare derived monthly features with the underlying daily and event records, then visualise utilisation and repayment behaviour. This is an evidence checkpoint: do not register a Feature View until the selected-account values are plausible.

In [ ]:
import matplotlib.pyplot as plt

default_accounts = session.table(f"{database_name}.{raw_schema}.FACT_DEFAULT_EVENT").select("ACCOUNT_ID").limit(2)
current_accounts = accounts.join(default_accounts, "ACCOUNT_ID", how="left_anti").select("ACCOUNT_ID").limit(2)
selected_accounts = default_accounts.union_all(current_accounts)

selected_raw_daily = (daily
                        .join(selected_accounts, "ACCOUNT_ID")
                        .filter(sf.col("SNAPSHOT_DATE") >= sf.dateadd("day", sf.lit(-120), sf.lit(config["data"]["drift_start_date"])))
                        .select("ACCOUNT_ID", "SNAPSHOT_DATE", "OUTSTANDING_BALANCE", "CREDIT_LIMIT", "DAYS_PAST_DUE")
                        .sort("ACCOUNT_ID", "SNAPSHOT_DATE")
                    )
selected_derived = (balance_df
                        .join(selected_accounts, "ACCOUNT_ID")
                        .join(delinquency_df, ["ACCOUNT_ID"], how="inner")
                        .filter(sf.col("BALANCE_TS") == sf.col("DELINQUENCY_TS"))
                        .select("ACCOUNT_ID", "BALANCE_TS"
                                , "CURRENT_UTILISATION", "UTILISATION_STDDEV_90D"
                                , "BALANCE_CHANGE_30D", "CURRENT_DAYS_PAST_DUE"
                                , "DELINQUENCY_EPISODES_6M")
                        .sort("ACCOUNT_ID", "BALANCE_TS")
                    )
selected_payments = (payment_df
                        .join(selected_accounts, "ACCOUNT_ID")
                        .select("ACCOUNT_ID", "PAYMENT_TS"
                                , "PAYMENT_SUM_30D", "PAYMENT_TO_BALANCE_RATIO_30D"
                                , "PAYMENT_AMOUNT_TREND_3M", "MISSED_PAYMENT_COUNT_3M")
                        .sort("ACCOUNT_ID", "PAYMENT_TS")
                    )
selected_raw_profile = (accounts
                            .join(selected_accounts, "ACCOUNT_ID")
                            .join(customers, "CUSTOMER_ID")
                            .select("ACCOUNT_ID", "CUSTOMER_ID", "OPEN_DATE"
                                    , "ORIGINAL_CREDIT_LIMIT", "CUSTOMER_SINCE_DATE")
                            .sort("ACCOUNT_ID")
                        )
selected_raw_financials = (financials
                            .join(accounts.select("ACCOUNT_ID", "CUSTOMER_ID"), "CUSTOMER_ID")
                            .join(selected_accounts, "ACCOUNT_ID")
                            .select("ACCOUNT_ID", "SNAPSHOT_DATE", "MONTHLY_INCOME_ESTIMATE")
                            .sort("ACCOUNT_ID", "SNAPSHOT_DATE")
                        )
selected_raw_limit_events = (limit_events
                                .join(selected_accounts, "ACCOUNT_ID")
                                .select("ACCOUNT_ID", "EVENT_DATE", "OLD_LIMIT", "NEW_LIMIT")
                                .sort("ACCOUNT_ID", "EVENT_DATE")
                            )
selected_profile = (profile_df
                        .join(selected_accounts, "ACCOUNT_ID")
                        .select("ACCOUNT_ID", "PROFILE_TS"
                                , "CURRENT_CREDIT_LIMIT", "CURRENT_INCOME_ESTIMATE"
                                , "MONTHS_SINCE_LIMIT_CHANGE", "LIMIT_CHANGE_PERCENT_12M")
                        .sort("ACCOUNT_ID", "PROFILE_TS")
                    )
selected_contacts = (contact_df
                        .join(selected_accounts, "ACCOUNT_ID")
                        .select("ACCOUNT_ID", "CONTACT_TS"
                                , "CONTACT_COUNT_30D", "CONTACT_COUNT_90D"
                                , "BROKEN_PROMISE_COUNT_90D")
                        .sort("ACCOUNT_ID", "CONTACT_TS")
                    )
selected_raw_payments = (payments
                            .join(selected_accounts, "ACCOUNT_ID")
                            .select("ACCOUNT_ID", "PAYMENT_DATE", "PAYMENT_AMOUNT", "PAYMENT_STATUS")
                            .sort("ACCOUNT_ID", "PAYMENT_DATE")
                        )
selected_payment_reconciliation = (payment_events
                                    .join(selected_accounts, "ACCOUNT_ID")
                                    .select(observation_dates["ACCOUNT_ID"]
                                            , observation_dates["FEATURE_DATE"]
                                            , payments["PAYMENT_DATE"]
                                            , payments["PAYMENT_AMOUNT"]
                                            , payments["PAYMENT_STATUS"]
                                            , sf.datediff("day", payments["PAYMENT_DATE"]
                                                        , observation_dates["FEATURE_DATE"]).alias("DAYS_BEFORE_OBSERVATION"))
                                    .sort("ACCOUNT_ID", "FEATURE_DATE", "PAYMENT_DATE")
                                )
selected_raw_contacts = (contacts
                            .join(selected_accounts, "ACCOUNT_ID")
                            .select("ACCOUNT_ID", "CONTACT_TIMESTAMP", "CONTACT_REASON", "CONTACT_OUTCOME")
                            .sort("ACCOUNT_ID", "CONTACT_TIMESTAMP")
                        )
selected_raw_daily.show(20)
selected_raw_profile.show(20)
selected_raw_financials.show(20)
selected_raw_limit_events.show(20)
selected_profile.show(20)
selected_derived.show(20)
selected_raw_payments.show(20)
selected_payment_reconciliation.show(20)
selected_payments.show(20)
selected_raw_contacts.show(20)
selected_contacts.show(20)

plot_data = selected_derived.to_pandas()
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for account_id, cohort in plot_data.groupby("ACCOUNT_ID"):
    axes[0].plot(cohort["BALANCE_TS"], cohort["CURRENT_UTILISATION"], label=account_id)
    axes[1].plot(cohort["BALANCE_TS"], cohort["DELINQUENCY_EPISODES_6M"], label=account_id)
axes[0].set_ylabel("Current utilisation")
axes[1].set_ylabel("Delinquency episodes, 6 months")
axes[1].set_xlabel("Feature timestamp")
axes[0].legend(ncol=2)
plt.tight_layout()

The daily source is end-of-day state, so it becomes available at midnight on the following day. Payments and contacts on the observation date are excluded. Null recency and trend values mean the required prior event does not exist; zero event counts mean the window was observed but contained no qualifying event.

## 5. Register only the reviewed definitions

Feature View versions are notebook decisions, not `project.yaml` settings. Increment `feature_version` rather than overwriting a stored definition.

In [ ]:
feature_version = "V01"
feature_specs = [
    ("ACCOUNT_PROFILE_FV", profile_df, "PROFILE_TS", "Point-in-time account and customer profile."),
    ("ACCOUNT_BALANCE_FV", balance_df, "BALANCE_TS", "Current and trailing balance and utilisation behaviour."),
    ("ACCOUNT_PAYMENT_FV", payment_df, "PAYMENT_TS", "Trailing payment behaviour from posted and missed payment events."),
    ("ACCOUNT_CONTACT_FV", contact_df, "CONTACT_TS", "Trailing servicing contact behaviour."),
    ("ACCOUNT_DELINQUENCY_FV", delinquency_df, "DELINQUENCY_TS", "Current and trailing delinquency state."),
]
aggregation_view.attach_feature_desc({
    "AVG_UTILISATION_30D": "Mean daily utilisation over the trailing 30-day window.",
    "AVG_UTILISATION_90D": "Mean daily utilisation over the trailing 90-day window.",
    "MAX_UTILISATION_90D": "Maximum daily utilisation over the trailing 90-day window.",
    "AVG_UTILISATION_365D": "Mean daily utilisation over the trailing 365-day window.",
    "MAX_UTILISATION_365D": "Maximum daily utilisation over the trailing 365-day window.",
    "MAX_DAYS_PAST_DUE_3M": "Maximum days past due in the trailing 90-day window.",
    "MAX_DAYS_PAST_DUE_6M": "Maximum days past due in the trailing 180-day window.",
    "MAX_DAYS_PAST_DUE_365D": "Maximum days past due over the trailing 365-day window.",
    "DAYS_OVER_LIMIT_30D": "Number of days balance exceeded limit in the trailing 30-day window.",
    "DELINQUENT_DAYS_90D": "Days with a positive past-due balance in the trailing 90-day window.",
})
feature_descriptions = {
    "ACCOUNT_PROFILE_FV": {
        "PRODUCT_CODE": "Credit product family; REVOLVING_CREDIT for this portfolio.",
        "ORIGINATION_CHANNEL": "Channel through which the facility was opened.",
        "ACCOUNT_AGE_MONTHS": "Whole months from facility opening to feature timestamp.",
        "CUSTOMER_TENURE_MONTHS": "Whole months from customer relationship start to feature timestamp.",
        "CURRENT_CREDIT_LIMIT": "Credit limit effective at the feature timestamp, in account currency units.",
        "CURRENT_INCOME_ESTIMATE": "Most recent monthly income estimate available at the feature timestamp.",
        "LIMIT_TO_INCOME_RATIO": "Effective credit limit divided by monthly income estimate.",
        "MONTHS_SINCE_LIMIT_CHANGE": "Whole months since the latest prior credit-limit change; null when none exists.",
        "LIMIT_CHANGE_PERCENT_12M": "Relative size of the latest limit change in the preceding 12 months; zero when none exists.",
    },
    "ACCOUNT_BALANCE_FV": {
        "CURRENT_BALANCE": "Outstanding balance at the feature timestamp, in account currency units.",
        "CURRENT_UTILISATION": "Outstanding balance divided by effective credit limit.",
        "UTILISATION_STDDEV_90D": "Standard deviation of daily utilisation over the trailing 90-day window.",
        "BALANCE_CHANGE_30D": "Current balance less the balance 30 calendar days earlier.",
        "BALANCE_CHANGE_90D": "Current balance less the balance 90 calendar days earlier.",
        "AVAILABLE_CREDIT_RATIO": "Available credit divided by effective credit limit.",
    },
    "ACCOUNT_PAYMENT_FV": {
        "PAYMENT_SUM_30D": "Payment amount posted in the trailing 30-day window.",
        "PAYMENT_SUM_90D": "Payment amount posted in the trailing 90-day window.",
        "PAYMENT_COUNT_90D": "Number of scheduled payment events in the trailing 90-day window.",
        "PAYMENT_TO_BALANCE_RATIO_30D": "Trailing 30-day payments divided by the balance available before observation.",
        "PAYMENT_TO_AMOUNT_DUE_RATIO_30D": "Trailing 30-day payments divided by the amount due available before observation.",
        "MISSED_PAYMENT_COUNT_3M": "Missed payment events in the trailing 90-day window.",
        "MISSED_PAYMENT_COUNT_6M": "Missed payment events in the trailing 180-day window.",
        "DAYS_SINCE_LAST_PAYMENT": "Calendar days since the latest positive payment; null when none exists.",
        "PAYMENT_AMOUNT_TREND_3M": "Mean payment in the most recent 30 days less the mean from days 60 through 89.",
    },
    "ACCOUNT_CONTACT_FV": {
        "CONTACT_COUNT_30D": "Servicing contacts in the trailing 30-day window.",
        "CONTACT_COUNT_90D": "Servicing contacts in the trailing 90-day window.",
        "BROKEN_PROMISE_COUNT_90D": "Contacts ending in BROKEN_PROMISE in the trailing 90-day window.",
    },
    "ACCOUNT_DELINQUENCY_FV": {
        "CURRENT_DAYS_PAST_DUE": "Days past due at the feature timestamp.",
        "DELINQUENCY_EPISODES_6M": "Transitions from current to delinquent in the trailing 180-day window.",
        "CONSECUTIVE_DELINQUENT_MONTHS": "Approximate completed 30-day delinquency periods at the feature timestamp.",
        "DAYS_SINCE_LAST_DELINQUENCY": "Calendar days since the most recent delinquent day; null when none exists.",
    },
}
feature_views = []
aggregation_exists = fs.list_feature_views().filter(
    (sf.col("NAME") == "ACCOUNT_AGGREGATION_FV") & (sf.col("VERSION") == feature_version)
).count() > 0
if not aggregation_exists:
    fs.register_feature_view(aggregation_view, version=feature_version, overwrite=False)
feature_views.append(fs.get_feature_view("ACCOUNT_AGGREGATION_FV", feature_version))

for name, feature_df, timestamp_col, description in feature_specs:
    feature_view = FeatureView(
        name=name,
        entities=[credit_account],
        feature_df=feature_df,
        timestamp_col=timestamp_col,
        refresh_freq=None,
        desc=description,
    )
    feature_view.attach_feature_desc(feature_descriptions[name])
    exists = fs.list_feature_views().filter(
        (sf.col("NAME") == name) & (sf.col("VERSION") == feature_version)
    ).count() > 0
    if not exists:
        fs.register_feature_view(feature_view, version=feature_version, overwrite=False)
    stored_view = fs.get_feature_view(name, feature_version)
    print(name, stored_view.query)
    feature_views.append(stored_view)
fs.list_feature_views().show()

## 6. Validate registered values, then choose temporal boundaries

First retrieve all six registered Feature Views for the reviewed facilities, including native 365-day aggregations. Then use only finalised labels to define Datasets. The drift start remains untouched hold-out. The development boundary is chosen here after inspecting pre-holdout monthly support; the 90-day finality condition creates an explicit purge around each cutoff.

Now that all six Feature Views are registered, retrieve their values for the same selected facilities. This includes the native 365-day aggregation outputs and provides the final raw-to-Feature-Store comparison before choosing temporal Dataset boundaries.

In [ ]:
selected_feature_spine = (observation_dates
                            .join(selected_accounts, "ACCOUNT_ID")
                            .select("ACCOUNT_ID"
                                    , sf.col("FEATURE_DATE").cast("timestamp_ntz").alias("OBSERVATION_TS"))
                        )
selected_feature_values = fs.retrieve_feature_values(
    spine_df=selected_feature_spine,
    features=feature_views,
    spine_timestamp_col="OBSERVATION_TS",
    include_feature_view_timestamp_col=True,
    join_method="cte",
)
selected_feature_values.select("ACCOUNT_ID", "OBSERVATION_TS"
                                , "CURRENT_CREDIT_LIMIT", "CURRENT_INCOME_ESTIMATE"
                                , "CONTACT_COUNT_90D", "PAYMENT_SUM_30D"
                                , "AVG_UTILISATION_365D", "MAX_UTILISATION_365D"
                                , "MAX_DAYS_PAST_DUE_365D"
                            ).sort("ACCOUNT_ID", "OBSERVATION_TS").show(20)

In [ ]:
labels = session.table(f"{database_name}.{raw_schema}.AVAILABLE_GROUND_TRUTH")
label_coverage = (labels
                    .group_by("OBSERVATION_DATE")
                    .agg(sf.count("ACCOUNT_ID").alias("OBSERVATION_COUNT")
                        , sf.max("OUTCOME_FINALITY_DATE").alias("FINALITY_DATE"))
                    .sort("OBSERVATION_DATE")
                )
label_coverage.show(40)

development_cutoff = "2025-07-01"
holdout_cutoff = config["data"]["drift_start_date"]
labelled_spine = labels.select("ACCOUNT_ID"
                                , sf.col("OBSERVATION_DATE").cast("timestamp_ntz").alias("OBSERVATION_TS")
                                , "OUTCOME_FINALITY_DATE"
                                , "DEFAULT_WITHIN_90D")

development_spine = labelled_spine.filter((sf.col("OBSERVATION_TS") < development_cutoff)
                                            & (sf.col("OUTCOME_FINALITY_DATE") < development_cutoff))
validation_spine = labelled_spine.filter((sf.col("OBSERVATION_TS") >= development_cutoff)
                                            & (sf.col("OBSERVATION_TS") < holdout_cutoff)
                                            & (sf.col("OUTCOME_FINALITY_DATE") < holdout_cutoff))
print("Development rows:", development_spine.count())
print("Validation rows:", validation_spine.count())

## 7. Generate immutable point-in-time Datasets

The labelled spine controls prediction time. Each Feature View must return values no later than `OBSERVATION_TS`. The Dataset version is unique and shared by development and validation names for a clear experiment hand-off.

In [ ]:
dataset_version = "D_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid4().hex[:6].upper()
development_dataset = fs.generate_dataset(
    name=f"{database_name}.{dev_schema}.CREDIT_DEVELOPMENT",
    version=dataset_version,
    spine_df=development_spine,
    features=feature_views,
    spine_timestamp_col="OBSERVATION_TS",
    spine_label_cols=["DEFAULT_WITHIN_90D"],
    include_feature_view_timestamp_col=True,
    join_method="cte",
    desc="Embargoed development observations from reviewed Feature Views.",
)
validation_dataset = fs.generate_dataset(
    name=f"{database_name}.{dev_schema}.CREDIT_VALIDATION",
    version=dataset_version,
    spine_df=validation_spine,
    features=feature_views,
    spine_timestamp_col="OBSERVATION_TS",
    spine_label_cols=["DEFAULT_WITHIN_90D"],
    include_feature_view_timestamp_col=True,
    join_method="cte",
    desc="Embargoed temporal validation observations from reviewed Feature Views.",
)
development_df = development_dataset.read.to_snowpark_dataframe(only_feature_cols=False)
validation_df = validation_dataset.read.to_snowpark_dataframe(only_feature_cols=False)
print("Dataset version for notebook 03:", dataset_version)
development_df.sort("OBSERVATION_TS", "ACCOUNT_ID").show(10)

In [ ]:
retrieved = development_df.union_all(validation_df)
timestamp_columns = [column for column in retrieved.columns if column.endswith("_TS") and column != "OBSERVATION_TS"]
late_feature_condition = sf.lit(False)
for timestamp_column in timestamp_columns:
    late_feature_condition = late_feature_condition | (sf.col(timestamp_column) > sf.col("OBSERVATION_TS"))
late_feature_count = retrieved.filter(late_feature_condition).count()
spine_count = development_spine.count() + validation_spine.count()
retrieved_count = retrieved.count()
distinct_key_count = retrieved.select("ACCOUNT_ID", "OBSERVATION_TS").distinct().count()
print("Late feature timestamps:", late_feature_count)
print("Spine / retrieved / distinct keys:", spine_count, retrieved_count, distinct_key_count)
if late_feature_count or not (spine_count == retrieved_count == distinct_key_count):
    raise ValueError("Investigate point-in-time retrieval before modelling.")
print("Use this immutable version in notebook 03:", dataset_version)

## 8. Inspect the Feature View to Dataset hand-off

The stored Dataset should identify the Feature Views used to create it. Inspect both the API-level Feature View list and upstream lineage before passing the immutable version to notebook 03.

In [ ]:
development_feature_views = fs.load_feature_views_from_dataset(development_dataset)
validation_feature_views = fs.load_feature_views_from_dataset(validation_dataset)
print("Development Feature Views:", [(view.name, view.version) for view in development_feature_views])
print("Validation Feature Views:", [(view.name, view.version) for view in validation_feature_views])
display(development_dataset.lineage(direction="upstream"))
display(validation_dataset.lineage(direction="upstream"))
print("Dataset version for notebook 03:", dataset_version)